## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set your project path
PROJECT_PATH = '/content/drive/MyDrive/vindr-spinexr'

import os
os.chdir(PROJECT_PATH)
print(f"Working directory: {os.getcwd()}")

## Step 2: Check GPU

In [ ]:
!nvidia-smi

## Step 3: Install Dependencies

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")

In [ ]:
# Install detectron2
!pip install 'git+https://github.com/facebookresearch/detectron2.git@4841e70ee48da72c32304f9ebf98138c2a70048d'

In [ ]:
# Install other dependencies
!pip install timm pycocotools scikit-learn pandas pydot

## Step 4: Verify Installation

In [ ]:
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer
from detectron2.config import get_cfg

print(f"Detectron2 version: {detectron2.__version__}")
print("✓ Detectron2 installed successfully!")

## Step 5: Verify Data Files ✓

In [ ]:
import os
import pandas as pd
import glob

# Check data structure
print("Checking data files...")
print(f"✓ Train annotations: {os.path.exists('data/annotations/train.csv')}")
print(f"✓ Train images dir: {os.path.exists('data/train_pngs')}")
print(f"✓ Config file: {os.path.exists('spine/configs/sparsercnn_improved.yaml')}")
print(f"✓ Pretrained weights: {os.path.exists('pretrained/r101_100pro_3x_model.pth')}")

# Count images
num_images = len(glob.glob('data/train_pngs/*.png'))
print(f"\n✓ Found {num_images} training images")

# Load and check annotations
train_df = pd.read_csv('data/annotations/train.csv')
print(f"✓ Total annotations: {len(train_df)}")
print(f"✓ Unique images: {train_df['image_id'].nunique()}")
print(f"\nLesion distribution:")
print(train_df['lesion_type'].value_counts())

print("\n🎉 All data verified! Ready to train!")

## Step 6: Start Training 🚀

In [ ]:
# Train the model
!python spine/train_net.py \
    --num-gpus 1 \
    --config-file spine/configs/sparsercnn_improved.yaml \
    OUTPUT_DIR outputs/sparsercnn_improved

## Step 7: Monitor Training (Optional)

In [ ]:
# View training logs
!tail -n 50 outputs/sparsercnn_improved/log.txt

## Step 8: Evaluate Final Model

In [ ]:
# Evaluate the final model
!python spine/train_net.py \
    --eval-only \
    --num-gpus 1 \
    --config-file spine/configs/sparsercnn_improved.yaml \
    MODEL.WEIGHTS outputs/sparsercnn_improved/model_final.pth

## Step 9: Compare with Paper Baseline

In [ ]:
import json

# Load metrics
with open('outputs/sparsercnn_improved/metrics.json', 'r') as f:
    metrics = [json.loads(line) for line in f]

# Get final mAP
final_metrics = metrics[-1]
final_map = final_metrics.get('bbox/AP50', 0)

print("=" * 50)
print("FINAL RESULTS")
print("=" * 50)
print(f"Paper baseline (Sparse R-CNN): 33.15 mAP@0.5")
print(f"Our improved model:             {final_map:.2f} mAP@0.5")
print(f"Improvement:                    +{final_map - 33.15:.2f} mAP")
print("=" * 50)

if final_map > 33.15:
    print("\n🎉 SUCCESS! We beat the paper baseline!")
    print(f"Achievement: {final_map:.2f} mAP@0.5 (target was 36-38)")
else:
    print("\n⚠️ Did not beat baseline yet.")
    print("Current: {:.2f} vs Target: 36-38 mAP@0.5".format(final_map))

## Step 10: Results Summary

In [ ]:
print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)
print("\nResults saved to Google Drive:")
print(f"  {PROJECT_PATH}/outputs/sparsercnn_improved/")
print("\nFiles:")
print("  • model_final.pth - Trained model weights")
print("  • metrics.json - Training metrics")
print("  • log.txt - Full training log")
print("\n✓ All results automatically synced to your Google Drive!")
print("\nYou can close this notebook - results are saved!")